# Динамическая модель автопоезда и полный цикл MPC

Документ объясняет модель тягача с полуприцепом из
[dmpc_solver.py](dmpc_solver.py), класс `DMPCSolverV2`, и вспомогательные
вычисления из [base.py](base.py), класс `CasadiBase`.

Формулы записаны в Markdown с математическими блоками `$$ ... $$` и строчными
выражениями `$ ... $`. Для их отображения нужен Markdown-preview с поддержкой
LaTeX/MathJax/KaTeX.

> Ниже разделены математический вывод и фактическая реализация. Два расхождения
> между ними перечислены в разделе 8. Это результаты аналитического сравнения,
> а не исправления кода или подтверждённые испытаниями дефекты.

## 1. Общий путь от модели до управления

1. Получить текущее состояние автопоезда, целевую траекторию и параметры машины.
2. По состоянию вычислить скорости осей и углы увода шин.
3. По углам увода, нагрузкам и сцеплению вычислить боковые силы.
4. Сформировать матрицу инерции $M$, вектор внешних обобщённых сил $Q$ и
   вектор скоростных членов $k$.
5. Вычислить производную состояния и интегрировать её для прогноза движения.
6. Найти будущие управления, минимизирующие ошибки и резкость руления при
   выполнении динамики и ограничений.
7. В стандартном цикле MPC применить ближайшую часть плана, получить новое
   фактическое состояние и повторить оптимизацию.

```text
Текущее состояние + целевая траектория + параметры
                         |
                         v
           Оптимизатор выбирает пробный план руления
                         |
                         v
      Скорости осей -> углы увода -> силы шин -> M, Q, k
                         |
                         v
          Интегрирование -> прогноз состояний -> стоимость
                         |
                         v
           Производные -> поправка к плану -> повтор
                         |
                         v
                Найденный план руления
```

Пробные планы вычисляются внутри решателя. На реальной машине их по очереди
не испытывают.

## 2. Состояние, управление и геометрия

### 2.1. Девять компонент состояния

$$
s=[X,Y,\psi_1,\psi_2,v_x,v_y,r_1,r_2,\delta]^T.
$$

| Обозначение | В коде | Смысл |
|---|---|---|
| $X,Y$ | первые две компоненты | Положение центра масс тягача в системе траектории |
| $\psi_1$ | `psi1` | Курс тягача |
| $\psi_2$ | `psi2` | Курс полуприцепа |
| $v_x,v_y$ | `v1x`, `v1y` | Скорость центра масс тягача в осях тягача |
| $r_1$ | `psi1_dot` | Угловая скорость тягача |
| $r_2$ | `psi2_dot` | Угловая скорость полуприцепа |
| $\delta$ | `fwsa` | Угол передних колёс, не рулевого колеса |

Управление — скорость изменения угла передних колёс:

$$
\boxed{u=\dot\delta}.
$$

Углы измеряются в радианах, скорости поворота — в рад/с.

Два независимых плоских твёрдых тела имеют шесть координат. Совпадение точки
сцепки даёт две геометрические связи, поэтому достаточно четырёх координат
$X,Y,\psi_1,\psi_2$. Для динамики нужны ещё четыре скорости. Угол колёс добавляет
девятую компоненту, поскольку управление задаёт его производную:

$$
4\text{ координаты}+4\text{ скорости}+1\text{ угол колёс}=9.
$$

Угол складывания $\eta=\psi_1-\psi_2$ вычисляется из состояния. Координаты и
скорости полуприцепа восстанавливаются через сцепку. Силы и ускорения здесь
не являются отдельными состояниями: они вычисляются из текущего состояния.

Это достаточный набор для данной плоской модели. Крены, подвеска, вращение
колёс, динамика деформации шин и подробная динамика рулевого привода отдельных
состояний здесь не имеют.

### 2.2. Геометрические параметры

| Параметр | Значение в используемой геометрии |
|---|---|
| $l_1$ | От центра масс тягача вперёд до передней оси |
| $l_2$ | От центра масс тягача назад до сцепки |
| $l_3$ | От центра масс тягача назад до задней оси |
| $b_1$ | От сцепки назад до центра масс полуприцепа |
| $b_3$ | От центра масс полуприцепа назад до его задней оси |
| $m_1,m_2$ | Массы тягача и полуприцепа с грузом |
| $I_1,I_2$ | Моменты инерции относительно вертикальных осей через центры масс |

Введём единичные векторы вдоль и поперёк тел:

$$
e_i=\begin{bmatrix}\cos\psi_i\\\sin\psi_i\end{bmatrix},
\qquad
n_i=\begin{bmatrix}-\sin\psi_i\\\cos\psi_i\end{bmatrix}.
$$

$$
\dot e_i=r_i n_i,\qquad \dot n_i=-r_i e_i.
$$

Положения сцепки и центра масс полуприцепа:

$$
p_H=p_1-l_2e_1,\qquad
\boxed{p_2=p_1-l_2e_1-b_1e_2}.
$$

## 3. Скорости полуприцепа и системы координат

### 3.1. Скорость центра масс

Дифференцируем положение:

$$
v_2=v_1-l_2r_1n_1-b_1r_2n_2
=v_xe_1+(v_y-l_2r_1)n_1-b_1r_2n_2.
$$

При $\eta=\psi_1-\psi_2$:

$$
e_2^Te_1=\cos\eta,\quad e_2^Tn_1=-\sin\eta,
\quad n_2^Te_1=\sin\eta,\quad n_2^Tn_1=\cos\eta.
$$

Проецирование на оси полуприцепа даёт:

$$
\boxed{v_{2x}=v_x\cos\eta-(v_y-l_2r_1)\sin\eta},
$$

$$
\boxed{v_{2y}=v_x\sin\eta+(v_y-l_2r_1)\cos\eta-b_1r_2}.
$$

Вращение полуприцепа не даёт продольной добавки: скорость от вращения точки
на его продольной оси направлена поперёк этой оси. Вращение тягача влияет на
продольную скорость полуприцепа через движение сцепки и угол складывания.

### 3.2. Скорость задней оси

Для $L=b_1+b_3$ положение задней оси:

$$
p_A=p_1-l_2e_1-Le_2.
$$

В системе **полуприцепа**:

$$
\boxed{v_{2rx}=v_x\cos\eta-(v_y-l_2r_1)\sin\eta},
$$

$$
\boxed{v_{2ry}=v_x\sin\eta+(v_y-l_2r_1)\cos\eta-Lr_2}.
$$

Это `v2rx`, `v2ry` в коде. Исходные $v_x,v_y$ относятся к тягачу, но результат
этих формул уже выражен в осях полуприцепа.

В системе **тягача** скорость той же точки:

$$
v_A^{(1)}=
\begin{bmatrix}
v_x-Lr_2\sin\eta\\
v_y-l_2r_1-Lr_2\cos\eta
\end{bmatrix}.
$$

Матрицы перехода для компонент вектора:

$$
R_{1\to2}=
\begin{bmatrix}\cos\eta&-\sin\eta\\\sin\eta&\cos\eta\end{bmatrix},
\qquad
R_{2\to1}=R_{1\to2}^T.
$$

Для преобразования скорости нужны обе компоненты. Одну поперечную компоненту
нельзя повернуть как полный вектор.

### 3.3. Мощность не нужно поворачивать

В системе полуприцепа боковая сила оси равна $F^{(2)}=[0,F_t]^T$. Её мощность:

$$
\mathcal P=(F^{(2)})^Tv_A^{(2)}=F_t v_{2ry}.
$$

Это скаляр. При одновременном повороте силы и скорости:

$$
(RF)^T(Rv)=F^TR^TRv=F^Tv.
$$

Поэтому мощность одинакова в обеих системах. Нельзя сначала умножить боковую
силу на поперечную скорость, а затем «повернуть» полученную мощность.

## 4. Шины: углы увода, нагрузки и насыщение сил

### 4.1. Углы увода

Угол увода $\alpha$ — разность направления движения точки колеса и направления
качения колеса. В модели:

$$
\alpha_f=\operatorname{atan2}(v_y+l_1r_1,v_x)-\delta,
$$

$$
\alpha_r=\operatorname{atan2}(v_y-l_3r_1,v_x),
\qquad
\alpha_t=\operatorname{atan2}(v_{2ry},v_{2rx}).
$$

Передние колёса повёрнуты относительно тягача на $\delta$, поэтому этот угол
вычитается. Для полуприцепа скорости уже заданы относительно его осей.

### 4.2. Сила каждой эквивалентной оси

$$
\boxed{F_i=\operatorname{clip}(-N_iC_i\alpha_i,-\mu N_i,+\mu N_i)},
\qquad i\in\{f,r,t\}.
$$

| Величина | Смысл |
|---|---|
| $N_i$ | Вертикальная нагрузка на ось, Н |
| $C_i$ | Нормированная на нагрузку боковая жёсткость, $1/\text{рад}$ |
| $N_iC_i$ | Боковая жёсткость оси, Н/рад |
| $\alpha_i$ | Угол увода, рад |
| $\mu$ | Коэффициент сцепления |

В `get_normal_force` сначала вычисляется `c_alpha = vertical_load * stiffness_coef`.
Поэтому параметр `C_alpha_*` в этой реализации — именно нормированный коэффициент.
Минус означает противодействие силы уводу.

$$
\operatorname{clip}(z,L,U)=
\begin{cases}
L,&z<L,\\z,&L\le z\le U,\\U,&z>U.
\end{cases}
$$

Например, при $N=40000$ Н, $C=10$ рад$^{-1}$, $\mu=0.8$ предел равен $32000$ Н.
При $\alpha=0.02$ рад сила равна $-8000$ Н. При $\alpha=0.1$ рад линейная
формула даёт $-40000$ Н, но после `clip` получается $-32000$ Н.

### 4.3. Откуда берутся вертикальные нагрузки

`calculate_axle_vertical_loads` использует статическое распределение веса.
Обозначим $L_1=l_1+l_3$, $L_2=b_1+b_3$, а $x_t,x_1,x_H$ — значения параметров
`semitrailer_com_position_x`, `truck_com_position_x`, `hitch_point_position_x`
в принятой этой функцией координатной конвенции. Тогда код вычисляет:

$$
N_H=\frac{m_2gx_t}{L_2},\qquad N_t=m_2g-N_H,
$$

$$
N_r=\frac{N_H(L_1-x_H)+m_1g(L_1-x_1)}{L_1},
\qquad N_f=m_1g+N_H-N_r.
$$

Это статические нагрузки; динамический перенос веса через отдельную модель
подвески здесь не рассчитывается.

### 4.4. Насыщение не запрещает занос

`clip` срабатывает внутри каждого вычисления динамики. Он ограничивает силу,
которая попадёт в прогноз, но не запрещает большой угол увода.

Оптимизатор видит последствия: дальнейшее увеличение увода не создаёт
неограниченно большую силу и не обеспечивает произвольную кривизну движения.
Однако он может выбрать режим насыщения, если остальные ограничения это допускают.

Отдельного ограничения $|\alpha_i|\le\alpha_{\max}$ в рассматриваемой постановке
нет. Если нужен запас до насыщения, можно было бы добавить, например:

$$
|N_iC_i\alpha_i|\le\rho\mu N_i,\qquad 0<\rho<1.
$$

Это пример возможного расширения, не существующее ограничение кода.
Ограничивать силу после `clip` величиной $\mu N_i$ бессмысленно: это уже выполнено.

## 5. Матрица инерции M

Введём вектор независимых скоростей:

$$
\nu=[v_x,v_y,r_1,r_2]^T.
$$

Кинетическая энергия:

$$
T=\frac12m_1(v_x^2+v_y^2)+\frac12I_1r_1^2
+\frac12m_2(v_{2x}^2+v_{2y}^2)+\frac12I_2r_2^2.
$$

Положим $w=v_y-l_2r_1$. Тогда:

$$
v_{2x}^2+v_{2y}^2
=(v_x\cos\eta-w\sin\eta)^2
+(v_x\sin\eta+w\cos\eta-b_1r_2)^2
$$

$$
=v_x^2+w^2-2b_1r_2(v_x\sin\eta+w\cos\eta)+b_1^2r_2^2.
$$

После раскрытия скобок:

$$
\begin{aligned}
T={}&\tfrac12(m_1+m_2)(v_x^2+v_y^2)
+\tfrac12(I_1+m_2l_2^2)r_1^2
+\tfrac12(I_2+m_2b_1^2)r_2^2\\
&-m_2l_2v_yr_1-m_2b_1\sin\eta\,v_xr_2
-m_2b_1\cos\eta\,v_yr_2
+m_2l_2b_1\cos\eta\,r_1r_2.
\end{aligned}
$$

Из записи $T=\frac12\nu^TM\nu$ получаем:

$$
\boxed{
M=
\begin{bmatrix}
m_1+m_2&0&0&-m_2b_1\sin\eta\\
0&m_1+m_2&-m_2l_2&-m_2b_1\cos\eta\\
0&-m_2l_2&I_1+m_2l_2^2&m_2l_2b_1\cos\eta\\
-m_2b_1\sin\eta&-m_2b_1\cos\eta&m_2l_2b_1\cos\eta&I_2+m_2b_1^2
\end{bmatrix}}.
$$

Внедиагональные элементы описывают связь движений. Например, член
$-m_2l_2v_yr_1$ в энергии даёт $M_{23}=M_{32}=-m_2l_2$.

Компактный способ построения:

$$
J_2=
\begin{bmatrix}
\cos\eta&-\sin\eta&l_2\sin\eta&0\\
\sin\eta&\cos\eta&-l_2\cos\eta&-b_1
\end{bmatrix},
$$

$$
\boxed{M=\operatorname{diag}(m_1,m_1,I_1,I_2)+m_2J_2^TJ_2}.
$$

Здесь $[v_{2x},v_{2y}]^T=J_2\nu$. Матрица совпадает с `build_m_matrix_eq`.

## 6. Вектор обобщённых сил Q

$Q$ — вектор, а не матрица:

$$
Q=[Q_x,Q_y,Q_{\psi_1},Q_{\psi_2}]^T.
$$

Первые компоненты имеют размерность силы, последние — момента. Угловые
компоненты относятся к независимым движениям всей связанной системы.

### 6.1. Вывод через мощность

Если скорость точки силы $v_P=J_P\nu$, то:

$$
\mathcal P=F_P^Tv_P=F_P^TJ_P\nu=Q_P^T\nu,
\qquad \boxed{Q_P=J_P^TF_P}.
$$

Для передней оси тягача:

$$
\mathcal P_f=-F_f\sin\delta\,v_x
+F_f\cos\delta\,(v_y+l_1r_1),
$$

$$
Q_f=[-F_f\sin\delta,F_f\cos\delta,l_1F_f\cos\delta,0]^T.
$$

Для задней оси тягача:

$$
\mathcal P_r=F_r(v_y-l_3r_1),\qquad
Q_r=[0,F_r,-l_3F_r,0]^T.
$$

Для оси полуприцепа считаем мощность в его системе:

$$
\mathcal P_t=F_t v_{2ry}
=F_t[v_x\sin\eta+(v_y-l_2r_1)\cos\eta-(b_1+b_3)r_2],
$$

$$
Q_t=[F_t\sin\eta,F_t\cos\eta,-l_2F_t\cos\eta,-(b_1+b_3)F_t]^T.
$$

Плечо $b_1+b_3$ появляется потому, что изменение $\psi_2$ при фиксированном
тягаче поворачивает полуприцеп вокруг сцепки. Это не отдельное уравнение
моментов полуприцепа относительно его центра масс.

Итого:

$$
\boxed{Q_{\mathrm{tires}}=
\begin{bmatrix}
-F_f\sin\delta+F_t\sin\eta\\
F_f\cos\delta+F_r+F_t\cos\eta\\
l_1F_f\cos\delta-l_3F_r-l_2F_t\cos\eta\\
-(b_1+b_3)F_t
\end{bmatrix}}.
$$

Силы идеальной сцепки внутренние: они противоположны и приложены к точкам
с одинаковой скоростью. Их суммарная мощность равна нулю, поэтому отдельно
в $Q$ они не входят.

### 6.2. Гравитация в плоскости движения

Пусть $g_{1x},g_{1y}$ и $g_{2x},g_{2y}$ — компоненты одной и той же плоской
проекции гравитации в системах тягача и полуприцепа. Из мощности
$m_1(g_{1x}v_x+g_{1y}v_y)+m_2(g_{2x}v_{2x}+g_{2y}v_{2y})$ следует:

$$
Q_g=
\begin{bmatrix}
m_1g_{1x}+m_2(g_{2x}\cos\eta+g_{2y}\sin\eta)\\
m_1g_{1y}+m_2(-g_{2x}\sin\eta+g_{2y}\cos\eta)\\
m_2l_2g_{2x}\sin\eta-m_2l_2g_{2y}\cos\eta\\
-m_2b_1g_{2y}
\end{bmatrix}.
$$

Полный вектор $Q=Q_{\mathrm{tires}}+Q_g$. Отличие гравитационной добавки в коде
описано в разделе 8.

## 7. Вектор скоростных членов k

### 7.1. Зачем он нужен

Даже для одного тягача дифференцирование $v_1=v_xe_1+v_yn_1$ даёт:

$$
a_1=(\dot v_x-r_1v_y)e_1+(\dot v_y+r_1v_x)n_1.
$$

Компоненты скорости заданы во вращающихся осях. Поэтому физическое ускорение
не равно просто $[\dot v_x,\dot v_y]^T$. В $k$ собираются скоростные члены,
а в $M\dot\nu$ — члены с производными независимых скоростей.

### 7.2. Вывод из энергии

Для краткости положим $m=m_1+m_2$, $l=l_2$, $b=b_1$, $c=\cos\eta$, $s_\eta=\sin\eta$.
Производные энергии:

$$
P_x=\frac{\partial T}{\partial v_x}=mv_x-m_2bs_\eta r_2,
$$

$$
P_y=\frac{\partial T}{\partial v_y}=mv_y-m_2lr_1-m_2bcr_2,
$$

$$
H_1=\frac{\partial T}{\partial r_1}
=(I_1+m_2l^2)r_1-m_2lv_y+m_2lbcr_2,
$$

$$
H_2=\frac{\partial T}{\partial r_2}
=(I_2+m_2b^2)r_2-m_2bs_\eta v_x-m_2bcv_y+m_2lbcr_1.
$$

$$
T_\eta=\frac{\partial T}{\partial\eta}
=-m_2bcv_xr_2+m_2bs_\eta v_yr_2-m_2lbs_\eta r_1r_2.
$$

С учётом вращающихся осей уравнения имеют вид:

$$
\begin{aligned}
\dot P_x-r_1P_y&=Q_x,\\
\dot P_y+r_1P_x&=Q_y,\\
\dot H_1+v_xP_y-v_yP_x-T_\eta&=Q_{\psi_1},\\
\dot H_2+T_\eta&=Q_{\psi_2}.
\end{aligned}
$$

При дифференцировании используем $\dot\eta=r_1-r_2$. Например:

$$
\dot P_x=m\dot v_x-m_2bs_\eta\dot r_2-m_2bc(r_1-r_2)r_2.
$$

После добавления $-r_1P_y$ смешанные члены $r_1r_2$ сокращаются:

$$
m\dot v_x-m_2bs_\eta\dot r_2
+m_2lr_1^2+m_2bcr_2^2-mr_1v_y=Q_x.
$$

Аналогично:

$$
\dot P_y=m\dot v_y-m_2l\dot r_1-m_2bc\dot r_2
+m_2bs_\eta(r_1-r_2)r_2.
$$

После добавления $r_1P_x$:

$$
m\dot v_y-m_2l\dot r_1-m_2bc\dot r_2
+mr_1v_x-m_2bs_\eta r_2^2=Q_y.
$$

Сбор всех четырёх уравнений даёт:

$$
\boxed{k=
\begin{bmatrix}
m_2l_2r_1^2+m_2b_1r_2^2\cos\eta-(m_1+m_2)r_1v_y\\
(m_1+m_2)r_1v_x-m_2b_1r_2^2\sin\eta\\
-m_2l_2(r_1v_x-b_1r_2^2\sin\eta)\\
-m_2b_1r_1[v_x\cos\eta+(l_2r_1-v_y)\sin\eta]
\end{bmatrix}}.
$$

Это результат вывода для указанной геометрии; отличие второй компоненты
от реализации приведено далее.

### 7.3. Общий алгоритм: производные матрицы и энергии

Для обычных обобщённых координат $q$, когда используемые скорости действительно
равны $\dot q$, кинетическая энергия имеет вид:

$$
T=\frac12\dot q^TM(q)\dot q.
$$

Первая производная энергии по скоростям даёт обобщённые импульсы, а вторая —
матрицу инерции:

$$
p=\frac{\partial T}{\partial\dot q}=M\dot q,
\qquad
M_{ij}=\frac{\partial^2T}{\partial\dot q_i\partial\dot q_j}.
$$

Таким образом, $M$ — гессиан энергии **по скоростям**, при фиксированных
координатах. Для одной точки это тот же переход:
$T=mv^2/2\to\partial T/\partial v=mv\to\partial^2T/\partial v^2=m$.

Если внешние силы, включая гравитацию, собраны в $Q$, уравнения Лагранжа дают:

$$
\frac{d}{dt}\frac{\partial T}{\partial\dot q}
-\frac{\partial T}{\partial q}=Q,
$$

$$
M\ddot q+\dot M\dot q-\frac{\partial T}{\partial q}=Q.
$$

Отсюда:

$$
\boxed{k=\dot M\dot q-\frac{\partial T}{\partial q}},
\qquad
\boxed{\dot M=\sum_i\frac{\partial M}{\partial q_i}\dot q_i}.
$$

Порядок вычислений:

1. Взять исходную матрицу $M$ и независимо дифференцировать её по каждой
   координате $q_i$. Каждый элемент дифференцируется на своём месте.
2. Каждую полученную матрицу умножить на соответствующую **скалярную скорость**
   $\dot q_i$ и сложить результаты. Получится матрица $\dot M$.
3. Умножить $\dot M$ на **весь вектор скоростей** $\dot q$. Получится вектор.
4. Вычесть вектор $\partial T/\partial q$, вычисленный при фиксированных скоростях.

Умножать нужно на скорости, а не на координаты. Дифференцирование по разным
координатам выполняется независимо от исходной $M$: последовательное
дифференцирование результата дало бы вторую смешанную производную, которая
для вычисления $\dot M$ здесь не нужна.

Например:

$$
M(q_1,q_2)=
\begin{bmatrix}q_1^2&q_1q_2\\q_1q_2&\cos q_2\end{bmatrix},
$$

$$
\frac{\partial M}{\partial q_1}=
\begin{bmatrix}2q_1&q_2\\q_2&0\end{bmatrix},
\qquad
\frac{\partial M}{\partial q_2}=
\begin{bmatrix}0&q_1\\q_1&-\sin q_2\end{bmatrix}.
$$

$$
\dot M=
\begin{bmatrix}
2q_1\dot q_1&q_2\dot q_1+q_1\dot q_2\\
q_2\dot q_1+q_1\dot q_2&-\sin q_2\,\dot q_2
\end{bmatrix}.
$$

Если переменная встречается в нескольких элементах, дифференцируются все эти
элементы. Не зависящие от неё элементы становятся нулями. Размер матрицы
сохраняется, элементы не удаляются.

### 7.4. Частный случай: скорости во вращающейся системе тягача

В нашей модели координаты $q=[X,Y,\psi_1,\psi_2]^T$, но энергия записана через
$\nu=[v_x,v_y,r_1,r_2]^T$. Эти скорости не равны $\dot q$, поскольку:

$$
\dot X=v_x\cos\psi_1-v_y\sin\psi_1,
\qquad
\dot Y=v_x\sin\psi_1+v_y\cos\psi_1.
$$

Поэтому нельзя просто заменить $\dot q$ на $\nu$ в общей формуле для $k$.
Гессиан $\partial^2T/\partial\nu^2$ по-прежнему даёт нашу матрицу инерции,
но в уравнениях движения появляются поправки из-за вращения осей.

Производная самой матрицы считается обычным способом. Она зависит только
от $\eta=\psi_1-\psi_2$, поэтому:

$$
\boxed{\dot M=\frac{\partial M}{\partial\eta}(r_1-r_2)}.
$$

Обозначим, как в разделе 7.2:

$$
P_x=\frac{\partial T}{\partial v_x},\qquad
P_y=\frac{\partial T}{\partial v_y},\qquad
T_\eta=\frac{\partial T}{\partial\eta}.
$$

Из уравнений раздела 7.2 и равенства
$\partial T/\partial\nu=M\nu$ получаем компактную формулу именно для нашей модели:

$$
\boxed{
k=\dot M\nu
+\begin{bmatrix}
-r_1P_y\\r_1P_x\\v_xP_y-v_yP_x\\0
\end{bmatrix}
+\begin{bmatrix}0\\0\\-T_\eta\\+T_\eta\end{bmatrix}
}.
$$

Первое слагаемое учитывает изменение матрицы инерции. Второе — вращение осей
тягача. Третье — угловую зависимость энергии; знаки следуют из
$\partial T/\partial\psi_1=T_\eta$ и
$\partial T/\partial\psi_2=-T_\eta$ при фиксированных $\nu$.

Например, первая компонента:

$$
(\dot M\nu)_1=-m_2b_1\cos\eta\,(r_1-r_2)r_2,
$$

$$
P_y=(m_1+m_2)v_y-m_2l_2r_1-m_2b_1\cos\eta\,r_2.
$$

Следовательно:

$$
\begin{aligned}
k_1&=(\dot M\nu)_1-r_1P_y\\
&=-m_2b_1\cos\eta\,r_1r_2+m_2b_1\cos\eta\,r_2^2\\
&\quad-(m_1+m_2)r_1v_y+m_2l_2r_1^2+m_2b_1\cos\eta\,r_1r_2\\
&=m_2l_2r_1^2+m_2b_1\cos\eta\,r_2^2-(m_1+m_2)r_1v_y.
\end{aligned}
$$

Смешанные члены $r_1r_2$ сократились. Здесь $k_1$ — первая компонента,
то есть `k[0]` в Python.

Почему нельзя забыть поправки, видно даже для одиночного тягача:

$$
T=\tfrac12m(v_x^2+v_y^2)+\tfrac12Ir_1^2,
\qquad M=\operatorname{diag}(m,m,I).
$$

Матрица постоянна и энергия не зависит от координат, но:

$$
m\dot v_x-mr_1v_y=F_x,\qquad
m\dot v_y+mr_1v_x=F_y.
$$

Значит, $k=[-mr_1v_y,mr_1v_x,0]^T$ не равен нулю. Эти члены появляются именно
из-за вращения осей. Если с самого начала выразить энергию через
$\dot X,\dot Y,\dot\psi_1,\dot\psi_2$, можно использовать общую формулу раздела 7.3,
но матрица инерции будет представлена в других переменных.

## 8. Математический вывод и текущая реализация

При сравнении с `_system_dynamic` на момент подготовки документа обнаружены
следующие различия, которые требуют отдельной проверки перед изменением модели.

| Место | Результат вывода | Текущий код |
|---|---|---|
| Вторая компонента $k$ (`k[1]`) | $-m_2b_1r_2^2\sin\eta+(m_1+m_2)r_1v_x$ | Вместо $r_2^2$ используется $r_1^2$ |
| Третья компонента $Q_g$ (`q_e[2]`) | $m_2l_2g_{2x}\sin\eta-m_2l_2g_{2y}\cos\eta$ | Первый член отсутствует |

Первое различие исчезает при $\sin\eta=0$ или $r_1^2=r_2^2$; второе — при
$\sin\eta=0$ или $g_{2x}=0$. Поэтому прямолинейный режим может их не выявить.

Документ не меняет исходники. Формулы вывода не следует автоматически считать
побитовым описанием фактического прогноза решателя.

## 9. Полное уравнение состояния и продольное движение

Код сначала решает:

$$
\widetilde d=M^{-1}(Q-k).
$$

Это математическая запись: реализация использует `ca.solve`, а не явное
построение обратной матрицы. Затем выполняет:

```python
ddq = ca.solve(M, q_e - k)
ddq[0] = target_acceleration
```

В результате фактическая производная состояния:

$$
\boxed{\dot s=f(s,u,a_{\mathrm{ref}},p)=
\begin{bmatrix}
v_x\cos\psi_1-v_y\sin\psi_1\\
v_x\sin\psi_1+v_y\cos\psi_1\\
r_1\\r_2\\a_{\mathrm{ref}}\\
\widetilde d_2\\\widetilde d_3\\\widetilde d_4\\u
\end{bmatrix}}.
$$

### 9.1. Почему моделируются преимущественно боковые силы

MPC подбирает руление при заданном скоростном профиле. Тяга и торможение не
являются его неизвестными. Их силы и продольное проскальзывание шин явно не
моделируются. Боковые силы при повёрнутых колёсах или сложенном автопоезде
всё равно имеют проекции на продольную ось тягача.

Замена `ddq[0]` — дополнительное упрощение. Для полной согласованной постановки
с заданным продольным ускорением следовало бы учитывать это ускорение при
решении остальных связанных уравнений и определять необходимую продольную силу.

Совместное использование сцепления для торможения и поворота здесь не
учитывается явно. Например, приближённое ограничение
$F_x^2+F_y^2\le(\mu N)^2$ отличается от реализованного $|F_y|\le\mu N$.

### 9.2. Где форма A x + B u

Модель нелинейна из-за тригонометрии, произведений скоростей, `atan2` и насыщения.
Правильная общая запись — $\dot s=f(s,u,a_{\mathrm{ref}},p)$.
При фиксированных внешних входах:

$$
\dot s=f_0(s)+Bu,\qquad B=[0,0,0,0,0,0,0,0,1]^T.
$$

Локально около опорного движения:

$$
\Delta\dot s\approx A(t)\Delta s+B\Delta u,
\qquad A=\left.\frac{\partial f}{\partial s}\right|_{\bar s,\bar u}.
$$

Но задача в коде использует нелинейную модель, а не одну постоянную матрицу $A$.

## 10. Дискретизация и постановка оптимизации

### 10.1. Шаг прогноза

Горизонт разбит на $N$ интервалов длиной $h$. На интервале управление $u_k$
постоянно. Интегрирование даёт:

$$
\boxed{s_{k+1}=F_h(s_k,u_k,a_k,p)}.
$$

Эйлер: $F_h=s_k+h f(s_k,u_k,a_k,p)$.
По умолчанию используется RK4:

$$
\begin{aligned}
K_1&=f(s_k,u_k,a_k,p),\\
K_2&=f(s_k+\tfrac h2K_1,u_k,a_k,p),\\
K_3&=f(s_k+\tfrac h2K_2,u_k,a_k,p),\\
K_4&=f(s_k+hK_3,u_k,a_k,p),\\
F_h&=s_k+\tfrac h6(K_1+2K_2+2K_3+K_4).
\end{aligned}
$$

При `substep_count > 1` интервал делится на подшаги и эта процедура повторяется.
`states_count = 30` по умолчанию означает 30 интервалов прогноза, а не 30
компонент состояния. Горизонт равен $Nh$.

Целевые точки далее нумеруются от 1 до $N$, по концам интервалов:

$$
a_k=
\begin{cases}
(v_{x,1}^{\mathrm{ref}}-v_{x,0})/h,&k=0,\\
(v_{x,k+1}^{\mathrm{ref}}-v_{x,k}^{\mathrm{ref}})/h,&k>0.
\end{cases}
$$

### 10.2. Неизвестные и ограничения динамики

Вектор неизвестных:

$$
z=[s_0^T,u_0,s_1^T,u_1,\ldots,u_{N-1},s_N^T]^T.
$$

Начальное состояние фиксируется измерением/оценкой. Для каждого интервала:

$$
s_0=s_{\mathrm{measured}},\qquad
F_h(s_k,u_k,a_k,p)-s_{k+1}=0.
$$

Это multiple shooting: будущие состояния — отдельные переменные, но равенства
не позволяют выбрать их независимо от физического прогноза.

При $N=30$ имеется $9+30(1+9)=309$ скалярных переменных, 270 равенств динамики
и фиксация девяти начальных компонент. Независимый выбор по смыслу — 30 управлений.

### 10.3. Стоимость движения

Для состояния на конце интервала и соответствующей целевой точки:

$$
e_\perp=-(X-X^{\mathrm{ref}})\sin\psi_1^{\mathrm{ref}}
+(Y-Y^{\mathrm{ref}})\cos\psi_1^{\mathrm{ref}},
\qquad e_\psi=\psi_1-\psi_1^{\mathrm{ref}}.
$$

Приближённые боковые ошибки бамперов тягача:

$$
e_f=e_\perp+L_fe_\psi,\qquad e_r=e_\perp-L_re_\psi,
$$

где $L_f=l_1+\text{front overhang}$, $L_r=l_3+\text{rear overhang}$.
Эти формулы используют линейное приближение по ошибке курса. Бамперы помогают
штрафовать неправильную ориентацию, даже если центр тягача находится на траектории.

Оценка бокового рывка:

$$
j_{\mathrm{approx}}=
\frac{(v_x^{\mathrm{ref}})^2u}{(l_1+l_3)\cos^2\delta}.
$$

Она соответствует производной $v^2\tan\delta/L$ по углу при фиксированной
скорости. Это не полная производная бокового ускорения динамической модели.

$$
\ell_k=w_k\left[
\frac{e_f^2}{\sigma_b^2}+\frac{e_r^2}{\sigma_b^2}
+\frac{e_\perp^2}{\sigma_c^2}
+\frac{u_k^2}{\sigma_u^2}
+\frac{j_{\mathrm{approx}}^2}{\sigma_j^2}
\right].
$$

| Масштаб | Параметр кода |
|---|---|
| $\sigma_b$ | `bumpers_weight` |
| $\sigma_c$ | `base_weight` |
| $\sigma_u$ | `regularization_weight` |
| $\sigma_j$ | `jerk_regularization_weight` |
| $w_k$ | `states_weights[k]` |

Параметры `*_weight` здесь стоят в знаменателях: увеличение масштаба ослабляет
соответствующий штраф. $w_k$, напротив, умножает всю стоимость шага.

Прямого штрафа за угол полуприцепа в этой функции нет. Он влияет на решение
через динамику. Не все восемь компонент целевой точки напрямую участвуют в стоимости.

### 10.4. Полная задача

$$
\boxed{
\begin{aligned}
\min_z\quad &J(z)=\sum_{k=0}^{N-1}\ell_k(s_{k+1},u_k)\\
\text{при}\quad&s_0=s_{\mathrm{measured}},\\
&s_{k+1}=F_h(s_k,u_k,a_k,p),\\
&u_{\min}\le u_k\le u_{\max},\\
&s_{\min}\le s_k\le s_{\max}.
\end{aligned}}
$$

Python-постановка ограничивает будущие состояния: $v_x\in[0,35]$ м/с,
$v_y\in[-35,35]$ м/с, $r_1,r_2\in[-9,9]$ рад/с, $|\delta|\le42^\circ$.
Управление ограничено `max_fwsa_rate`. Начальное состояние при подготовке
решения фиксируется равенством нижних и верхних границ фактическому состоянию.
Эти численные границы описывают постановку кода, а не доказанный безопасный режим.

Массы, инерции, сцепление и геометрия задают физическую реакцию. Масштабы штрафов
задают компромисс точности и плавности. Шаг и горизонт задают глубину и точность
прогноза. Обоснование конкретных настроек требует данных калибровки и испытаний.

## 11. Как решатель ищет управление

### 11.1. Это не случайный перебор

CasADi строит выражения динамики, стоимости и ограничений, а также предоставляет
производные. IPOPT ищет решение нелинейной задачи с ограничениями.

В `_prepare_solver_data` исходные управления заполняются переданным
`initial_fwsa_rate`; состояния приближаются целевыми точками, а углы колёс:

$$
\delta_k^{(0)}=\delta_0+kh\,u_{\mathrm{initial}}.
$$

Это начальная догадка, не план для отправки на автомобиль.

### 11.2. Как раннее руление влияет на позднюю ошибку

Для дискретной динамики:

$$
A_k=\frac{\partial F_h}{\partial s_k},\qquad
B_k=\frac{\partial F_h}{\partial u_k},
\qquad
\Delta s_{k+1}\approx A_k\Delta s_k+B_k\Delta u_k.
$$

Если мысленно исключить будущие состояния через динамику:

$$
\frac{\partial s_1}{\partial u_0}=B_0,\qquad
\frac{\partial s_2}{\partial u_0}=A_1B_0,\qquad
\frac{\partial s_3}{\partial u_0}=A_2A_1B_0.
$$

Решатель учитывает цепочку:

$$
u_0\to\delta\to F_y\to\text{ускорения}\to\text{будущее состояние}\to J.
$$

Соответствующая производная стоимости:

$$
\frac{dJ}{du_i}=
\frac{\partial\ell_i}{\partial u_i}
+\sum_{k=i}^{N-1}
\frac{\partial\ell_k}{\partial s_{k+1}}
\frac{\partial s_{k+1}}{\partial u_i}.
$$

В реализации состояния не исключаются: решатель корректирует их вместе
с управлениями, одновременно учитывая равенства динамики.

### 11.3. Что делает итерация IPOPT

Это не просто обновление $U\leftarrow U-\alpha\nabla J$.
Обозначим равенства динамики $c(z)=0$ и их якобиан $C=\partial c/\partial z$.
Блок для ограничения $F_h(s_k,u_k)-s_{k+1}=0$:

$$
\begin{bmatrix}A_k&B_k&-I\end{bmatrix}.
$$

Для объяснения временно опустим границы. Лагранжиан:

$$
\mathcal L(z,\lambda)=J(z)+\lambda^Tc(z).
$$

Ньютоновский шаг для условий стационарности схематически определяется системой:

$$
\begin{bmatrix}H&C^T\\C&0\end{bmatrix}
\begin{bmatrix}\Delta z\\\Delta\lambda\end{bmatrix}
=-
\begin{bmatrix}\nabla_z\mathcal L\\c(z)\end{bmatrix},
\qquad H=\nabla_{zz}^2\mathcal L.
$$

IPOPT учитывает границы методом внутренней точки. Схематическая барьерная
добавка для свободных переменных со строгими конечными границами:

$$
-\mu_B\sum_i[\log(z_i-z_{i,\min})+\log(z_{i,\max}-z_i)].
$$

$\mu_B$ — параметр барьера, не коэффициент сцепления. Полная система решателя
содержит дополнительные переменные и условия; фиксированные переменные
не описываются таким логарифмическим барьером.

После вычисления направления выбирается длина шага:

$$
z^{(i+1)}=z^{(i)}+\alpha\Delta z.
$$

Затем снова вычисляются нелинейные функции и производные. В коде задан лимит
50 итераций. Глобальный минимум и успешная сходимость не гарантируются.
`clip` также делает модель кусочно-гладкой: на границах насыщения обычная
гладкая интерпретация производных требует осторожности.

## 12. От результата MPC до команды грузовику

Решатель возвращает:

$$
U^*=[u_0^*,\ldots,u_{N-1}^*],\qquad
S^*=[s_1^*,\ldots,s_N^*].
$$

`psi1_dot` — угловая скорость корпуса тягача, а не команда рулению.
Оптимизированный `control` — скорость поворота передних колёс.

При постоянном $u_0^*$ на интервале:

$$
\boxed{\delta(t_0+\tau)=\delta(t_0)+u_0^*\tau}.
$$

Например, $\delta_0=0.02$ рад, $u_0^*=0.1$ рад/с, $h=0.1$ с дают
$\delta_1=0.03$ рад.

В стандартном MPC применяется ближайшая часть плана, после чего задача
пересчитывается по обновлённому состоянию. Это замыкает обратную связь и
позволяет корректировать ошибки прогноза.

**Граница разобранного кода:** `dmpc_solver.py` возвращает состояния и управления,
а не формирует CAN-пакет. Какая величина реально отправляется рулевому приводу,
как учитываются задержки, преобразование угла колёс в угол руля и обработка
неуспешного решения, определяется последующим C++-контуром. Этот путь в рамках
данного разбора не прослежен, поэтому его нельзя отождествлять с формулой
$\delta_1=\delta_0+h u_0^*$ без проверки вызывающего кода.

## 13. Расширение: MPC управляет ускорением

Этот раздел описывает возможное расширение, а не реализованное поведение
`DMPCSolverV2`. Вместо заданного из профиля ускорения оптимизатор выбирает
вторую компоненту управления:

$$
u_k=[\dot\delta_k,a_{\mathrm{cmd},k}]^T.
$$

### 13.1. Почему подробная модель двигателя необязательна

Если выход MPC — требуемое ускорение, преобразование этой команды в момент
двигателя, работу трансмиссии и торможение можно оставить нижележащему
продольному контуру. Для прогноза MPC нужна модель реакции **всего этого контура**,
а не обязательно отдельные модели двигателя и коробки передач.

При этом командное ускорение не равно фактическому мгновенно:

$$
a_{\mathrm{cmd}}\longrightarrow
\text{продольный контур и автомобиль}\longrightarrow a_{\mathrm{actual}}.
$$

Скорость рассчитывается через фактическое ускорение. Если здесь под $a$ понимается
производная выбранной продольной скорости $v$, то:

$$
\dot v=a_{\mathrm{actual}},\qquad
v_{k+1}\approx v_k+h a_{\mathrm{actual},k}.
$$

Равенство $\dot v=a_{\mathrm{cmd}}$ — идеальное приближение исполнения команды,
а не установленное свойство реальной машины.

### 13.2. Модель задержки и нарастания ускорения

Простая кандидатная модель реакции:

$$
\boxed{
\tau\dot a_{\mathrm{actual}}+a_{\mathrm{actual}}
=K a_{\mathrm{cmd}}(t-d)+b
}.
$$

| Параметр | Смысл |
|---|---|
| $d$ | Задержка исполнения команды, с |
| $\tau>0$ | Постоянная времени нарастания ускорения, с |
| $K$ | Коэффициент передачи команды |
| $b$ | Смещение ускорения, м/с² |

Эта модель приближённо описывает весь путь от команды до движения. Для её
построения не требуется отдельно знать КПД трансмиссии и характеристики двигателя.
Параметры оцениваются по данным — это идентификация системы.

Например, при $K=1$, $b=0$, $d=0$ явный шаг Эйлера:

$$
a_{k+1}=a_k+\frac{h}{\tau}(a_{\mathrm{cmd},k}-a_k),
\qquad v_{k+1}=v_k+h a_k.
$$

Это иллюстрация; при слишком крупном шаге Эйлер может плохо описывать даже
устойчивую непрерывную систему. Для постоянной команды без задержки ускорение
можно обновлять точно:

$$
a_{k+1}=e^{-h/\tau}a_k+
(1-e^{-h/\tau})(K a_{\mathrm{cmd},k}+b).
$$

Ускорение и скорость нужно интегрировать согласованно, например совместно
методом RK4. При наличии задержки прогноз также использует историю уже поданных
команд; задержку нельзя описать одним состоянием ускорения без дополнительного
представления памяти входа.

При добавлении одного состояния фактического ускорения вектор состояния
расширяется с девяти до десяти компонент. История команд для задержки учитывается
отдельно либо через дополнительные состояния выбранной аппроксимации.

### 13.3. Как определить модель по логам

1. Определить точку передачи $a_{\mathrm{cmd}}$: модель должна описывать именно
   тот контур, которому MPC будет выдавать команду.
2. Сопоставить временные ряды команд, скорости и ускорения с корректными
   временными метками и одинаковыми соглашениями о знаках и координатах.
3. Оценить $d,\tau,K,b$ по ошибке между измеренной реакцией и прогнозом модели.
4. Проверить прогноз на других записях, не использованных для подбора параметров,
   в том числе на горизонте, который будет использовать MPC.
5. Если одной модели недостаточно, разделить тягу и торможение или сделать
   параметры зависимыми от скорости, массы и других доступных признаков режима.

Если ускорение получают дифференцированием скорости, нужно учитывать шум и
задержку фильтрации. Также следует различать ускорение движения и сигнал датчика,
содержащий другие составляющие, включая влияние гравитации.

Идентифицированная модель действительна в проверенном диапазоне режимов.
Переключение передач, насыщение тяги и различия между разгоном и торможением
могут потребовать более сложного описания. Ограничения достижимого ускорения
можно задавать отдельно:

$$
a_{\min}(v,\ldots)\le a_{\mathrm{cmd}}\le a_{\max}(v,\ldots).
$$

### 13.4. Связь с динамикой автопоезда

Нужно договориться, что означает ускорение. Для скорости $v_x$ во вращающихся
осях тягача физическая продольная компонента ускорения равна:

$$
a_x=\dot v_x-r_1v_y.
$$

Если модель исполнения предсказывает $\dot v_x$, используем её выход напрямую.
Если она предсказывает $a_x$, то $\dot v_x=a_x+r_1v_y$.

Кроме того, заданную производную $v_x$ следует учитывать при решении остальных
связанных уравнений, а не заменять после решения полной системы. Обозначим
$w=[v_y,r_1,r_2]^T$ и $\beta=\dot v_x$. Тогда:

$$
\begin{bmatrix}M_{xx}&M_{xw}\\M_{wx}&M_{ww}\end{bmatrix}
\begin{bmatrix}\beta\\\dot w\end{bmatrix}
=\begin{bmatrix}Q_x-k_x\\Q_w-k_w\end{bmatrix},
$$

$$
\boxed{\dot w=M_{ww}^{-1}(Q_w-k_w-M_{wx}\beta)}.
$$

Такое использование заданного ускорения предполагает возможность обеспечить
необходимую дополнительную обобщённую силу только в первом уравнении.
Распределение тормозных сил по тягачу и полуприцепу и совместное использование
сцепления для продольных и боковых сил этим автоматически не моделируются.

### 13.5. Стоимость и исполнение плана

Когда ускорение становится свободным управлением, целевой скоростной профиль
нужно учитывать через стоимость или ограничения. Например:

$$
J=\sum_k\left[
\ell_{\mathrm{lateral},k}
+w_v(v_{k+1}-v^{\mathrm{ref}}_{k+1})^2
+w_a a_{\mathrm{cmd},k}^2
+w_j\left(\frac{a_{\mathrm{cmd},k}-a_{\mathrm{cmd},k-1}}h\right)^2
\right].
$$

Для первого шага используется предыдущая команда. Последний штраф ограничивает
резкость изменения команды; при динамике исполнения это не обязательно
фактический рывок автомобиля. Можно дополнительно штрафовать или ограничивать
именно прогнозируемый фактический рывок.

В существующей архитектуре `LongitudinalController` передаёт цели в
`ThrottleBrakeController`, а в обычном движении `FeedbackAccelerationController`
вычисляет командное ускорение с обратной связью. Затем `CarCommandProducer`
использует контроллеры силовой установки и ретардера и формирует команду.

При подключении нового MPC нужно согласовать целевые скорость, расстояние и
ускорение существующего контура с новым планом. Иначе обратная связь по старой
траектории может противодействовать выбранному MPC ускорению. Модель исполнения
нужно идентифицировать для фактически выбранного способа подключения.

## 14. Навигация по исходникам

| Этап | Файл и функция |
|---|---|
| Вертикальные нагрузки | [base.py](base.py), `calculate_axle_vertical_loads` |
| Сила шины и насыщение | [base.py](base.py), `get_normal_force` |
| Матрица инерции | [base.py](base.py), `build_m_matrix_eq` |
| Скорости осей, силы, Q, k, производная состояния | [dmpc_solver.py](dmpc_solver.py), `_system_dynamic` |
| Ошибки и стоимость | [dmpc_solver.py](dmpc_solver.py), `_objective_detailed` |
| Интегрирование | [dmpc_solver.py](dmpc_solver.py), `_integrate_euler`, `_integrate_rk4` |
| Функция шага | [dmpc_solver.py](dmpc_solver.py), `_build_integration_step` |
| Переменные и ограничения NLP | [dmpc_solver.py](dmpc_solver.py), `__build_nlp_problem` |
| Начальное приближение и фиксация состояния | [dmpc_solver.py](dmpc_solver.py), `_prepare_solver_data` |
| Вызов решателя и разбор результата | [dmpc_solver.py](dmpc_solver.py), `solve` |


# Model Predictive Path Integral Control (MPPI)

## 1. Что такое MPC

Пусть система описывается динамикой

$$
x_{t+1} = f(x_t, u_t),
$$

где:

- $x_t$ — состояние системы;
- $u_t$ — управление.

Например для машины:

$$
x_t =
\begin{pmatrix}
\text{position}\\
\text{velocity}\\
\text{orientation}
\end{pmatrix},
\qquad
u_t =
\begin{pmatrix}
\text{steering}\\
\text{throttle}
\end{pmatrix}.
$$

MPC выбирает не одно управление, а целую последовательность на горизонте $T$:

$$
U = (u_0, u_1, \dots, u_{T-1}).
$$

По этой последовательности через модель динамики строится предсказанная траектория:

$$
x_0 \to x_1 \to \dots \to x_T.
$$

После этого считается стоимость траектории:

$$
J(U)
=
\phi(x_T)
+
\sum_{t=0}^{T-1} q(x_t, u_t).
$$

Здесь:

- $q(x_t, u_t)$ — cost на текущем шаге;
- $\phi(x_T)$ — terminal cost.

Например:

$$
q(x_t,u_t)
=
\|x_t - x_t^{ref}\|_Q^2
+
\|u_t\|_R^2.
$$

То есть мы штрафуем:

1. отклонение от желаемой траектории;
2. слишком большое управление.

Задача MPC:

$$
U^* = \arg\min_U J(U).
$$

После нахождения $U^*$ применяется только первое управление:

$$
u_0^*.
$$

После этого система переходит в новое состояние, горизонт сдвигается, и задача решается заново.

Это называется **receding horizon control**.

---

# 2. Что такое MPPI

MPPI — это способ решать задачу MPC через sampling.

То есть задача остаётся той же:

$$
\min_U J(U),
$$

но вместо обычного численного optimizer используется множество случайных rollout'ов.

Главная идея:

$$
\text{берём текущий guess } U
\to
\text{добавляем шум}
\to
\text{оцениваем траектории}
\to
\text{сдвигаем } U \text{ в сторону хороших perturbations}.
$$

---

# 3. Начальная последовательность управлений

На первом шаге можно взять:

$$
U^{(0)}=(0,0,\dots,0).
$$

После первого шага обычно используется решение с предыдущего MPC cycle.

Если было

$$
U^*=
(u_0^*,u_1^*,\dots,u_{T-1}^*),
$$

то после применения $u_0^*$ новый guess:

$$
U_{\text{guess}}
=
(u_1^*,u_2^*,\dots,u_{T-1}^*,u_{\text{last}}).
$$

---

# 4. Sampling в MPPI

Для каждого rollout $k$ генерируется случайная последовательность perturbations:

$$
\varepsilon^{(k)}
=
(
\varepsilon_0^{(k)},
\dots,
\varepsilon_{T-1}^{(k)}
).
$$

Для скалярного управления:

$$
\varepsilon_t^{(k)}
\sim
\mathcal N(0,\sigma^2).
$$

Для векторного управления:

$$
\varepsilon_t^{(k)}
\sim
\mathcal N(0,\Sigma).
$$

Матрица $\Sigma$ задаёт масштаб шума для каждой компоненты управления.

Например:

$$
u_t=
\begin{pmatrix}
\text{steering}\\
\text{throttle}
\end{pmatrix}
$$

и

$$
\Sigma=
\begin{pmatrix}
0.1^2 & 0\\
0 & 20^2
\end{pmatrix}.
$$

---

# 5. Создание sampled control sequences

Для каждого rollout:

$$
U^{(k)}
=
U+\varepsilon^{(k)}.
$$

То есть:

$$
u_t^{(k)}
=
u_t+\varepsilon_t^{(k)}.
$$

Например:

$$
U=(0,0,0),
$$

$$
\varepsilon^{(1)}=(0.2,-0.1,0.4),
$$

тогда

$$
U^{(1)}=(0.2,-0.1,0.4).
$$

---

# 6. Что такое rollout

Rollout — это симуляция будущего при конкретной sampled последовательности управлений.

Для rollout $k$:

$$
x_0^{(k)}=x_{\text{current}},
$$

$$
x_{t+1}^{(k)}
=
f(
x_t^{(k)},
u_t+\varepsilon_t^{(k)}
).
$$

В результате получаем траекторию:

$$
x_0,
x_1^{(k)},
\dots,
x_T^{(k)}.
$$

Физически система эту траекторию не проходит.

Это только simulation внутри модели.

---

# 7. Стоимость rollout

Для каждой траектории считается cost:

$$
S_k
=
\phi(x_T^{(k)})
+
\sum_{t=0}^{T-1}
q(
x_t^{(k)},
u_t+\varepsilon_t^{(k)}
).
$$

$S_k$ показывает, насколько хорош rollout $k$.

Маленький $S_k$ — хорошая траектория.

Большой $S_k$ — плохая.

---

# 8. Веса rollout'ов

Каждому rollout присваивается вес:

$$
w_k
=
\frac{
\exp(-S_k/\lambda)
}{
\sum_j \exp(-S_j/\lambda)
}.
$$

Чем меньше $S_k$, тем больше $w_k$.

Параметр $\lambda$ задаёт, насколько сильно алгоритм концентрируется на лучших rollout'ах.

---

# 9. Главная формула MPPI

Для каждого момента времени $t$:

$$
u_t^{new}
=
u_t^{old}
+
\sum_{k=1}^{K}
w_k \varepsilon_t^{(k)}.
$$

То есть

$$
\Delta u_t
=
\sum_k
w_k
\varepsilon_t^{(k)}.
$$

И затем:

$$
u_t^{new}
=
u_t^{old}
+
\Delta u_t.
$$

Смысл:

- $\varepsilon_t^{(k)}$ — предложение изменить управление;
- $w_k$ — насколько мы доверяем этому rollout;
- $\sum_k w_k \varepsilon_t^{(k)}$ — взвешенное среднее полезных изменений.

MPPI не выбирает один лучший rollout.

Он использует информацию от всех rollout'ов.

---

# 10. Полный pipeline MPPI

## Шаг 1

Получаем текущее состояние:

$$
x_{\text{current}}.
$$

## Шаг 2

Берём текущую nominal sequence:

$$
U=(u_0,\dots,u_{T-1}).
$$

## Шаг 3

Генерируем $K$ случайных perturbation sequences:

$$
\varepsilon^{(1)},
\dots,
\varepsilon^{(K)}.
$$

## Шаг 4

Создаём sampled controls:

$$
U^{(k)}=U+\varepsilon^{(k)}.
$$

## Шаг 5

Для каждого $U^{(k)}$ симулируем rollout:

$$
x_{t+1}^{(k)}
=
f(
x_t^{(k)},
u_t+\varepsilon_t^{(k)}
).
$$

## Шаг 6

Для каждого rollout считаем cost:

$$
S_k.
$$

## Шаг 7

Считаем веса:

$$
w_k
\propto
e^{-S_k/\lambda}.
$$

## Шаг 8

Обновляем всю последовательность:

$$
u_t
\leftarrow
u_t
+
\sum_k
w_k
\varepsilon_t^{(k)}.
$$

Для всех

$$
t=0,\dots,T-1.
$$

## Шаг 9

Применяем только

$$
u_0.
$$

## Шаг 10

Получаем новое состояние системы.

## Шаг 11

Сдвигаем horizon:

$$
(u_0,u_1,\dots,u_{T-1})
\to
(u_1,u_2,\dots,u_{T-1},u_{\text{last}}).
$$

## Шаг 12

Повторяем MPPI заново.

---

# 11. MPC vs MPPI

Обычный MPC формулирует задачу:

$$
\min_U J(U).
$$

Обычно её решает численный optimizer:

$$
U
\to
U-\eta\nabla J(U).
$$

MPPI решает ту же задачу через sampling:

$$
U
\to
U+\varepsilon^{(k)}
\to
S_k
\to
w_k
\to
U+\sum_k w_k\varepsilon^{(k)}.
$$

Главное отличие:

$$
\boxed{\text{MPC — постановка задачи}}
$$

$$
\boxed{\text{MPPI — sampling-based способ решать MPC-задачу}}
$$

---

# 12. Самая короткая схема

$$
U
\rightarrow
U+\varepsilon^{(k)}
\rightarrow
\text{rollout}
\rightarrow
S_k
\rightarrow
w_k
\rightarrow
U_{\text{new}}.
$$

где

$$
U_{\text{new}}
=
U+
\sum_k
w_k\varepsilon^{(k)}.
$$

После этого:

$$
\text{execute }u_0
\to
\text{shift horizon}
\to
\text{repeat}.
$$